In [2]:
print('a')

a


In [3]:
pip install -q transformers datasets rouge-score sacrebleu torch

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 10.1 MB/s eta 0:00:00


In [11]:
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from transformers import pipeline
from rouge_score import rouge_scorer
from sacrebleu import corpus_bleu

ImportError: cannot import name 'pipeline' from 'transformers' (/usr/local/lib/python3.12/dist-packages/transformers/__init__.py)

In [7]:
import transformers
print(transformers.__version__)

4.57.3


In [4]:
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from transformers import pipeline
from rouge_score import rouge_scorer
from sacrebleu import corpus_bleu

DEVICE = 0 if torch.cuda.is_available() else -1

# =====================================================
# 1. LOAD SMALL PUBLIC DATASET (FAST)
# =====================================================
print("Loading dataset...")
dataset = load_dataset(
    "amazon_reviews_multi",
    "en",
    split="train[:500]"  # 🔥 ONLY 500 rows
)

texts = dataset["review_body"]
refs  = dataset["review_title"]

# =====================================================
# 2. MODELS TO COMPARE
# =====================================================
models = {
    "PEGASUS": "google/pegasus-cnn_dailymail",
    "BART": "facebook/bart-large-cnn",
    "FLAN-T5": "google/flan-t5-base"
}

# =====================================================
# 3. METRICS
# =====================================================
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

def evaluate(model_name, hf_name):
    print(f"\nEvaluating {model_name}...")

    summarizer = pipeline(
        "summarization",
        model=hf_name,
        device=DEVICE
    )

    preds = []
    rouge_scores = {"rouge1": [], "rouge2": [], "rougeL": []}

    for text, ref in tqdm(zip(texts, refs), total=len(texts)):
        if "t5" in hf_name.lower():
            text = "summarize: " + text

        summary = summarizer(
            text,
            max_length=40,
            min_length=10,
            do_sample=False
        )[0]["summary_text"]

        preds.append(summary)

        scores = scorer.score(ref, summary)
        for k in rouge_scores:
            rouge_scores[k].append(scores[k].fmeasure)

    bleu = corpus_bleu(preds, [refs]).score

    return {
        "Model": model_name,
        "ROUGE-1": np.mean(rouge_scores["rouge1"]),
        "ROUGE-2": np.mean(rouge_scores["rouge2"]),
        "ROUGE-L": np.mean(rouge_scores["rougeL"]),
        "BLEU": bleu
    }

# =====================================================
# 4. RUN EVALUATION
# =====================================================
results = []
for name, hf in models.items():
    results.append(evaluate(name, hf))

# =====================================================
# 5. RESULTS + BEST MODEL
# =====================================================
results_df = pd.DataFrame(results).sort_values("ROUGE-L", ascending=False)

print("\n=== MODEL COMPARISON RESULTS ===\n")
print(results_df)

best_model = results_df.iloc[0]["Model"]
print(f"\n🏆 BEST MODEL (by ROUGE-L): {best_model}")


ImportError: cannot import name 'pipeline' from 'transformers' (/usr/local/lib/python3.12/dist-packages/transformers/__init__.py)

In [12]:
# Step 1: Check if there's a name conflict
!ls | grep transformers

# If you see something like transformers.py or transformers.ipynb → rename or delete it!
# Example fix:
# !mv transformers.py transformers_backup.p

In [13]:
# Step 2: Reinstall transformers cleanly (this fixes 95% of such cases)
!pip uninstall transformers -y
!pip install transformers --no-cache-dir

Found existing installation: transformers 4.57.3
Uninstalling transformers-4.57.3:
  Successfully uninstalled transformers-4.57.3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 278.3 MB/s eta 0:00:00


In [1]:
# Step 3: Also reinstall related key packages (to ensure compatibility)
!pip install --upgrade accelerate datasets evaluate peft torch --no-cache-dir

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 156.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 226.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 146.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 181.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 174.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 124.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 239.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 158.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 256.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# Step 4: Restart the kernel/runtime
# In Colab/Jupyter: Runtime → Restart Runtime
# Then try importing again:
from transformers import pipeline
print("Success!")

Success!


In [5]:
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rouge_score import rouge_scorer
from sacrebleu import corpus_bleu

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =====================================================
# 1. LOAD DATASET (SCRIPT-FREE, GUARANTEED)
# =====================================================
print("Loading dataset...")
dataset = load_dataset(
    "cnn_dailymail",
    "3.0.0",
    split="test[:500]"   # 🔥 only 500 samples → fast
)

texts = dataset["article"]
refs  = dataset["highlights"]

# =====================================================
# 2. MODELS (ALREADY TRAINED FOR SUMMARIZATION)
# =====================================================
models = {
    "PEGASUS": "google/pegasus-cnn_dailymail",
    "BART-CNN": "facebook/bart-large-cnn",
    "FLAN-T5": "google/flan-t5-base"
}

# =====================================================
# 3. METRICS
# =====================================================
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

def evaluate_model(name, hf_model):
    print(f"\nEvaluating {name}...")

    tokenizer = AutoTokenizer.from_pretrained(hf_model)
    model = AutoModelForSeq2SeqLM.from_pretrained(hf_model).to(DEVICE)
    model.eval()

    preds = []
    rouge_scores = {"rouge1": [], "rouge2": [], "rougeL": []}

    for text, ref in tqdm(zip(texts, refs), total=len(texts)):
        if "t5" in hf_model.lower():
            text = "summarize: " + text

        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to(DEVICE)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_length=128,
                num_beams=4,
                do_sample=False
            )

        pred = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        preds.append(pred)

        scores = scorer.score(ref, pred)
        for k in rouge_scores:
            rouge_scores[k].append(scores[k].fmeasure)

    bleu = corpus_bleu(preds, [refs]).score

    return {
        "Model": name,
        "ROUGE-1": np.mean(rouge_scores["rouge1"]),
        "ROUGE-2": np.mean(rouge_scores["rouge2"]),
        "ROUGE-L": np.mean(rouge_scores["rougeL"]),
        "BLEU": bleu
    }

# =====================================================
# 4. RUN EVALUATION
# =====================================================
results = []
for name, model_id in models.items():
    results.append(evaluate_model(name, model_id))

# =====================================================
# 5. RESULTS
# =====================================================
results_df = pd.DataFrame(results).sort_values("ROUGE-L", ascending=False)

print("\n=== MODEL COMPARISON RESULTS ===\n")
print(results_df)

print(f"\n🏆 BEST MODEL (by ROUGE-L): {results_df.iloc[0]['Model']}")


Loading dataset...


README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

100%|██████████| 500/500 [08:41<00:00,  1.04s/it]



=== MODEL COMPARISON RESULTS ===

      Model   ROUGE-1   ROUGE-2   ROUGE-L       BLEU
0   PEGASUS  0.353274  0.152244  0.264691  11.471177
1  BART-CNN  0.350642  0.147596  0.250806  10.521015
2   FLAN-T5  0.227270  0.080497  0.174088   6.006176

🏆 BEST MODEL (by ROUGE-L): PEGASUS
